In [5]:
""
import os

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents.middleware import SummarizationMiddleware
import base64

load_dotenv()

# 模型,系统提示词,工具, 记忆,会话管理, 图片识别

qwen = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen3.6-plus",
    temperature=0.7,
)


SYSTEM_PROMPT = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：

识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。

智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。

多维度评估与排序：从营养价值与制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名单。

结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。
请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥
"""

search_tool = TavilySearch(
    max_results=5,
    topic="general",  # general, news, finance
)

@tool
def web_search(query : str):
    """进行web搜索"""
    return search_tool.invoke(input=query)


# 连接sqlite
connection = sqlite3.connect("resources/checkpoint.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()

# 定义ThreadId
configId = {"configurable": {"thread_id": "1"}}

summarizationMiddleware = SummarizationMiddleware(
    model=qwen,  #指定模型
    trigger=("fraction",0.8), #触发器, 当消息达到3条时触发,这里是人消息数+AI回复消息数
    keep=("fraction",0.5), #压缩后保留的消息数
)

image_path = r"C:\Users\tim\Desktop\test.jpg"
with open(image_path, "rb") as image_file:
    image_base64 = base64.b64encode(image_file.read()).decode("utf-8")

message = HumanMessage([
                {"type": "text", "text": "识别该图片"},
                {
                    "type": "image",
                    "base64": image_base64,
                    "mime_type": "image/jpeg",
                },
            ])



agent = create_agent(
    model=qwen,
    tools=[search_tool],
    system_prompt=SYSTEM_PROMPT,
    middleware=[summarizationMiddleware],
    checkpointer=checkpointer,
)

# 阻塞式调用
response = agent.invoke(input={"messages": [message]},config=configId)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '识别该图片'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/jpeg'}]
================================ Human Message =================================

[{'type': 'text', 'text': '识别该图片'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/jpeg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_f70ca98076cd4c7b9e564472)
 Call ID: call_f70ca98076cd4c7b9e564472
  Args:
    query: 紫茄子 红辣椒 西兰花 绿甜椒 黄甜椒 红甜椒 绿叶蔬菜 葱 菜谱
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search

{"query": "紫茄子 红辣椒 西兰花 绿甜椒 黄甜椒 红甜椒 绿叶蔬菜 葱 菜谱", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.youtube.com/watch?v=EYMBNuH6kjE", "title": "黃、綠、紫，五色繽紛時蔬，\"彩色甜椒炒茄子\"。 - YouTube", "content": "【煮婦上菜】：紅、橙、黃、綠、紫，五色繽紛時蔬，\"彩色甜椒炒茄子\"。 401 view